<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/chat_with_multiple_Docs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community

In [ ]:
!pip install unstructured

In [ ]:
!pip install "unstructured[pdf]"

In [ ]:
!sudo apt-get update

In [ ]:
!sudo apt-get install poppler-utils

In [ ]:
!sudo apt-get install libleptonica-dev tesseract-ocr libtesseract-dev python3-pil tesseract-ocr-eng tesseract-ocr-script-latn

In [ ]:
!pip install unstructured-pytesseract
!pip install tesseract-ocr

In [ ]:
!pip install "unstructured[pptx]"

In [ ]:
!pip install langchain-astradb

In [ ]:
!pip install langchain  langchain-google-genai datasets pypdf


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
loader = DirectoryLoader('/content/Docs')

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)

In [ ]:
docs = loader.load_and_split(text_splitter=splitter)

In [ ]:
len(docs)

In [ ]:
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

In [ ]:
import os
from google.colab import userdata
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [ ]:
ASTRA_DB_API_ENDPOINT="https://00e83b42-4e26-4680-8d48-d8e5edaa55a5-us-east-2.apps.astra.datastax.com"
ASTRA_DB_APPLICATION_TOKEN="Api_token"
ASTRA_DB_KEYSPACE="default_keyspace"

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_astradb import AstraDBVectorStore

In [ ]:
vstore = AstraDBVectorStore(
    embedding=embedding,
    collection_name="multidoc_vector",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=ASTRA_DB_KEYSPACE,
)

In [ ]:
inserted_ids = vstore.add_documents(docs)

In [ ]:
print(f"\nInserted {len(inserted_ids)} documents.")

In [ ]:
prompt_template = """
You are an AI philosopher drawing insights from the roadmap of "rag," "llama3," and "genai."
Craft thoughtful answers based on this roadmap, mixing and matching existing paths.
Your responses should be concise and strictly related to the provided context.

ROADMAP CONTEXT:
{context}

QUESTION: {question}

YOUR ANSWER:"""

In [ ]:
prompt_template = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
retriever = vstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
retriever

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GEMINI_API_KEY)

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
chain.invoke("can you tell me the Nike Air Force 1 price and color")

In [ ]:

chain.invoke("can you tell me about me the objective of the clubshere-ai centralised platform")

In [ ]:
chain.invoke("explain the introduction of the Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks")